In [0]:
-- Cria a tabela de oportunidades (pipeline de vendas) a partir de dados gerados (substitui se já existir)
CREATE OR REPLACE TABLE ai_bi_data_analyst.ai_bi_dashboards.ca_opportunities AS
WITH
-- Lista com todos os IDs de clientes
cust AS (
  SELECT collect_list(customerid) AS ids
  FROM ai_bi_data_analyst.ai_bi_dashboards.ca_customers
),
-- Lista com todos os IDs de produtos
prod AS (
  SELECT collect_list(productid) AS ids
  FROM ai_bi_data_analyst.ai_bi_dashboards.ca_products
),
-- Gera 342 oportunidades com valores aleatórios (sementes fixas deixam o resultado reproduzível)
base AS (
  SELECT
    CONCAT('CA', CAST(428100 + id AS STRING)) AS opportunityid,                              -- ID único: CA428101 a CA428442
    element_at(cust.ids, 1 + CAST(FLOOR(rand(12) * size(cust.ids)) AS INT)) AS customerid,  -- cliente sorteado de ca_customers
    element_at(prod.ids, 1 + CAST(FLOOR(rand(13) * size(prod.ids)) AS INT)) AS productid,   -- produto sorteado de ca_products
    DATE_ADD(DATE'2024-06-19', CAST(FLOOR(rand(14) * 196) AS INT)) AS `date`,               -- data entre 2024-06-19 e 2024-12-31
    CAST(FLOOR(20 + rand(15) * rand(16) * 6000) AS DECIMAL(12,2)) AS amount,                -- valor entre 20 e 6020, com mais valores pequenos
    element_at(array('Shalini','Ingrid','Ethan','Sam','Gowoon','Hannah'),
               1 + CAST(FLOOR(rand(17) * 6) AS INT)) AS salesrep,                           -- vendedor sorteado
    rand(18) AS r                                                                           -- número aleatório usado para definir a fase
  FROM range(1, 343)
  CROSS JOIN cust
  CROSS JOIN prod
)
SELECT
  b.opportunityid,
  b.customerid,
  b.productid,
  c.province,                      -- província vem do cadastro do cliente, então sempre bate com ca_customers
  b.`date`,
  b.amount,
  b.salesrep,
  CASE                             -- fase do funil: 45% / 20% / 15% / 20%
    WHEN b.r < 0.45 THEN '0-Prospecting'
    WHEN b.r < 0.65 THEN '1-Active'
    WHEN b.r < 0.80 THEN '2-Closed Won'
    ELSE '3-Closed Lost'
  END AS phase
FROM base b
JOIN ai_bi_data_analyst.ai_bi_dashboards.ca_customers c
  ON b.customerid = c.customerid;

-- Confere: total de oportunidades (esperado: 342) e quantas há em cada fase
SELECT phase, COUNT(*) AS qtd
FROM ai_bi_data_analyst.ai_bi_dashboards.ca_opportunities
GROUP BY phase
ORDER BY phase;